# 🔄 Retries — Fault-Tolerant Nodes with RetryPolicy

## Learning Objectives
In this notebook, you will learn:
1. **`RetryPolicy`** — configuring automatic retries for flaky operations
2. **Backoff strategies** — exponential backoff with jitter for resilient APIs
3. **Targeted retries** — retrying only on specific exception types

## Prerequisites
- `langgraph` installed
- Understanding of nodes and graph building (notebooks `02`–`08`)

---
## 🔧 Part 1: Environment Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import random
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy

print("✅ Imports loaded successfully!")

---
## 📋 Part 2: Define State & Custom Exception

In [ ]:
# ============================================================================
# STATE & EXCEPTION DEFINITIONS
# ============================================================================
class WeatherState(TypedDict):
    city: str
    temperature: float
    conditions: str


class APIError(Exception):
    """Simulated API Error."""
    pass

print("✅ State and exception defined!")

---
## ⚙️ Part 3: Define the Nodes

### 3.1 Flaky API Node

Simulates an external API with a **70% failure rate**. When the retry policy
catches the `APIError`, it will automatically retry up to `max_attempts` times.

In [ ]:
# ============================================================================
# NODE: Simulated flaky weather API (70% failure rate)
# ============================================================================
def fetch_weather(state: WeatherState) -> WeatherState:
    """
    Simulates calling an external weather API.
    Will randomly fail to demonstrate retry behavior.
    """
    city = state["city"]

    if random.random() < 0.7:
        print(f"❌ API call failed for {city}")
        raise APIError(f"Weather API timeout for {city}")

    print(f"✅ Successfully fetched weather for {city}")
    temp = round(random.uniform(15, 30), 1)
    conditions = random.choice(["Sunny", "Cloudy", "Rainy", "Partly Cloudy"])

    return {
        "temperature": temp,
        "conditions": conditions
    }

### 3.2 Formatter Node

In [ ]:
# ============================================================================
# NODE: Format the weather result
# ============================================================================
def format_result(state: WeatherState):
    """Format the weather data for display."""
    print(f"\n🌤️ Weather Report for {state['city']}:")
    print(f"Temperature: {state['temperature']} degrees")
    print(f"Conditions: {state['conditions']}")
    return state

print("✅ Nodes defined!")

---
## 🔗 Part 4: Build the Graph with RetryPolicy

The `RetryPolicy` is attached to a specific node via `add_node(..., retry_policy=...)`.

### RetryPolicy Parameters:
- **`max_attempts`** — maximum number of retry attempts
- **`initial_interval`** — wait time (seconds) before first retry
- **`backoff_factor`** — multiplier applied to the interval after each retry
- **`max_interval`** — upper bound on wait time between retries
- **`jitter`** — adds randomness to prevent thundering herd
- **`retry_on`** — the exception type(s) to catch and retry

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Attach RetryPolicy to flaky node
# ============================================================================
builder = StateGraph(WeatherState)

builder.add_node(
    "fetch_weather",
    fetch_weather,
    retry_policy=RetryPolicy(
        max_attempts=5,           # Try up to 5 times
        initial_interval=1.0,     # Wait 1s before first retry
        backoff_factor=2.0,       # Double the wait each time
        max_interval=10.0,        # Cap wait at 10s
        jitter=True,              # Add randomness to prevent thundering herd
        retry_on=APIError         # Only retry on this specific exception
    )
)

builder.add_node("format_result", format_result)

builder.add_edge(START, "fetch_weather")
builder.add_edge("fetch_weather", "format_result")
builder.add_edge("format_result", END)

graph = builder.compile()

print("✅ Graph compiled with RetryPolicy!")

---
## 🚀 Part 5: Run the Graph

We wrap the invocation in `try/except` since all retry attempts may be
exhausted if the simulated API keeps failing.

In [ ]:
# ============================================================================
# EXECUTION: Run with retry protection
# ============================================================================
try:
    result = graph.invoke({
        "city": "San Francisco",
        "temperature": 0.0,
        "conditions": ""
    })
    print(f"\n✨ Final Result: {result}")

except Exception as e:
    print(f"\n💥 All retry attempts exhausted: {e}")

---
## 📝 Summary

In this notebook, we learned:

### 1. RetryPolicy
- Attached per-node via `add_node(name, fn, retry_policy=...)`
- Handles transient failures automatically

### 2. Backoff Strategy
- **Exponential backoff** — `initial_interval * backoff_factor^attempt`
- **Jitter** — adds randomness to prevent synchronized retries
- **Max interval** — caps the maximum wait time

### 3. Targeted Exception Handling
- `retry_on=APIError` — only retries on specific exceptions
- Other exceptions propagate immediately

### Key Takeaway
RetryPolicy makes LangGraph nodes resilient to transient failures (API timeouts,
rate limits, network issues) without cluttering your node logic with retry code.